In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .appName("Final_mini")\
        .getOrCreate()

print("Spark Session created.")

Spark Session created.


In [4]:
from google.colab import files
uploaded=files.upload()

Saving sales.csv to sales.csv
Saving employees.csv to employees.csv
Saving employees_nested.json to employees_nested.json
Saving departments.csv to departments.csv


In [3]:
!ls

departments.csv  employees.csv	employees_nested.json  sales.csv  sample_data


In [14]:
df=spark.read.csv("employees.csv", header=True, inferSchema=True)
df.show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



In [ ]:
df_dept=spark.read.csv("departments.csv", header=True, inferSchema=True)
df_dept.show()

In [ ]:
df_nest=spark.read.option("multiline", "true").json("employees_nested.json")
df_nest.show()

In [ ]:
from pyspark.sql.functions import col
df_sales=spark.read.csv("sales.csv", header=True, inferSchema=True)
df_sales.show()


In [17]:
from pyspark.sql.functions import sum, max, min, avg, col
top_emp=df.orderBy(col("salary").desc()).first()
top_sal= top_emp["salary"]
top_name= top_emp["name"]


In [19]:
avg_sal= df.select(avg("salary").alias("Avg_salary")).first()["Avg_salary"]


In [20]:
dept_count= df.groupby("department").count().select(col("department"),col("count").alias("emp_count")).collect()


In [21]:
from pyspark.sql.functions import sum
total_sales=df_sales.agg(sum("amount").alias("Total_sales")).first()["Total_sales"]

In [23]:
best_sales_emp= df.join(df_sales, on="emp_id", how="inner")\
                .groupby("emp_id","name")\
                .agg(sum("amount").alias("Total_amount"))\
                .orderBy(col("Total_amount").desc()).first()

best_sales_name = best_sales_emp["name"]
best_sales_amt = best_sales_emp["Total_amount"]

In [24]:
report_lines = [
    "FINAL REPORT",
    f"Top Employee (Highest Salary): {top_emp} -> {top_sal}",
    f"Average Salary: {avg_sal}",
    "",
    "Department Employee Count:"
]

for row in dept_count:
    report_lines.append(f"{row['department']} -> {row['emp_count']}")

report_lines.append("")
report_lines.append(f"Total Sales: {total_sales}")
report_lines.append(f"Best Sales Employee: {best_sales_name} -> {best_sales_amt}")

df_report = spark.createDataFrame([(line,) for line in report_lines], ["value"])
df_report.coalesce(1).write.mode("overwrite").text("final_report.txt")

print("final_report.txt created successfully!")

final_report.txt created successfully!
